# Session 2C: MCP Client
**Goal**: Show 'USB-C for AI'. Tools are auto-discovered from a server without writing @tool locally.

In [ ]:
import os
import sys
sys.path.insert(0, '..')
import utils.bootstrap

from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from utils.llm_router import get_routed_llm

### Discover Tools & Run Agent (Async)

In [ ]:
import asyncio

async def run_mcp():
    MCP_SERVER_SCRIPT = os.path.abspath('mcp_server.py')
    client = MultiServerMCPClient({'inbox_server': {'command': sys.executable, 'args': [MCP_SERVER_SCRIPT], 'transport': 'stdio'}})
    tools = await client.get_tools()
    print(f"\ud83d\udd0d Discovered {len(tools)} tools via MCP!")
    
    llm = get_routed_llm(role='worker_model')
    agent = create_react_agent(llm, tools)
    
    result = await agent.ainvoke({"messages": [HumanMessage(content="Fetch my emails, analyze them, and schedule any meetings.")]})
    for msg in result['messages']:
        print(f"\n[{msg.type.upper()}]: {str(msg.content)[:300]}")
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  \ud83d\udd27 Tool Call: {tc['name']}({tc['args']})")

await run_mcp()